# Single File Monthly Statistics & Feature Engineering

**Project**: FLAPS  
**Dataset**: A **single** daily weather data from BOM in csv format  
**Purpose**: Compute monthly aggregated weather statistics and engineer features that are relevant for predicting monthly flight delay rates. Only works for one csv file at a time to demonstrate the feature engineering in more details.

---

## Table of Contents
1. [Setup](#1-setup)
2. [Load Cleaned Data](#2-load-cleaned-data)
3. [Basic Monthly Statistics](#3-basic-monthly-statistics)
4. [Temperature Features](#4-temperature-features)
5. [Precipitation Features](#5-precipitation-features)
6. [Wind Features](#6-wind-features)
7. [Humidity Features](#7-humidity-features)
8. [Extreme Weather Events](#8-extreme-weather-events)
10. [Summary Statistics](#10-summary-statistics)
11. [Export Results](#11-export-results)

## 1. Setup

Import necessary packages.

In [1]:
import numpy as np
import pandas as pd

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

## 2. Load Cleaned Data

Load the cleaned daily weather data from the previous notebook. See _data_cleaning_single.ipynb_ for more details.

In [2]:
# Note: Choose one arbitrary file as an example
df = pd.read_csv('../data/raw/syd/sydney_airport_amo-202512.csv', skiprows=range(1,13), skipfooter=1, encoding='latin-1', engine='python')

# Apply same cleaning as in data_cleaning_single.ipynb
df.columns = ['station', 'date', 'evapotranspiration', 'rain', 'panevaporation', 'temp_max', 'temp_min', 'hum_max', 'hum_min', 'wind_speed', 'solar_rad']
df = df.drop('station', axis=1)
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')

# Convert numeric columns
numeric_cols = ['evapotranspiration', 'rain', 'panevaporation', 'wind_speed', 'solar_rad']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Extract year and month for grouping
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['year_month'] = df['date'].dt.to_period('M')

print(f"Data loaded successfully!")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Total records: {len(df)}")
print(f"\nFirst few rows:")
df.head()

Data loaded successfully!
Date range: 2025-12-01 to 2025-12-31
Total records: 31

First few rows:


,date,evapotranspiration,rain,panevaporation,temp_max,temp_min,hum_max,hum_min,wind_speed,solar_rad,year,month,year_month
0,2025-12-01,8.1,0.0,12.6,26.5,15.5,55,19,7.36,21.90,2025,12,2025-12
1,2025-12-02,6.4,0.0,9.4,21.1,13.4,66,43,8.10,31.22,2025,12,2025-12
2,2025-12-03,7.6,0.0,8.8,26.8,14.2,71,27,5.51,31.46,2025,12,2025-12
3,2025-12-04,9.4,0.0,8.4,32.5,17.7,77,19,5.98,31.45,2025,12,2025-12
4,2025-12-05,11.7,0.0,13.2,37.8,19.6,66,8,6.26,30.79,2025,12,2025-12


## 3. Basic Monthly Statistics

Compute basic monthly aggregations.

In [19]:
# Core monthly statistics
monthly_stats = df.groupby('year_month').agg({
    #'column_name':'agg_type' 

    # Number of days in the month
    'date': 'count',  

    # Temperature features
    'temp_max': ['max', 'mean'],
    'temp_min': ['min', 'mean'],
    
    # Rainfall features
    'rain': ['max', 'mean'],
    
    # Wind features
    'wind_speed': ['mean', 'max'],
    
    # Humidity features
    'hum_max': ['mean'],
    'hum_min': ['mean']
    
    
}).reset_index()

In [20]:
# Combine column names
columns_combined = []
kk = 1
for col in monthly_stats.columns.values:
    if col[1] == "":
        columns_combined.append(col[0])
    else:
        columns_combined.append("_".join(col))
    kk += 1

monthly_stats.columns = columns_combined

# Rename aggregate columns to be more descriptive
monthly_stats.rename(columns={
    'date': 'days_in_month',
    'rain_mean': 'avg_rainfall_per_day',
    'temp_max_max': 'max_temperature',
    'temp_max_mean': 'avg_max_temp',
    'temp_min_min': 'min_temperature',
    'temp_min_mean': 'avg_min_temp',
    'rain_max': 'max_daily_rainfall',
    'wind_speed_mean': 'avg_wind_speed',
    'wind_speed_max': 'max_wind_speed',
    'hum_max_mean': 'avg_max_humidity',
    'hum_min_mean': 'avg_min_humidity'
}, inplace=True)

print("Basic monthly statistics computed:")
print(monthly_stats)

Basic monthly statistics computed:
  year_month  date_count  max_temperature  avg_max_temp  min_temperature  \
0    2025-12          31             42.6         28.28             13.4   

   avg_min_temp  max_daily_rainfall  avg_rainfall_per_day  avg_wind_speed  \
0         18.33                 5.8                  0.69            6.02   

   max_wind_speed  avg_max_humidity  avg_min_humidity  
0             8.1             80.87             38.26  


## 4. Temperature Features

Engineer temperature-related features that may affect flight operations. The rationale here is months with more volatile fluctuations are more likely to have flight delays:
* Average daily fluctuations
* Number of days with extremely high temperature (>35 C)
* Average day-to-day change of max temp

In [21]:
# Temperature feature engineering
# .agg() is not sufficient as derived time-series are involved (e.g. min-max range)
temp_features = df.groupby('year_month').apply(lambda x: pd.Series({
    # Temperature range (daily variability)
    'temp_range_mean': (x['temp_max'] - x['temp_min']).mean(),
    
    # Extreme temperature days
    'days_above_35C': (x['temp_max'] > 35).sum(),
    
    # Temperature volatility (day-to-day changes)
    'temp_volatility': x['temp_max'].diff().abs().mean()
})).reset_index()

# Left-join merge with monthly stats
monthly_stats = monthly_stats.merge(temp_features, on='year_month', how='left')

print("Temperature features added:")
print(temp_features)

Temperature features added:
  year_month  temp_range_mean  days_above_35C  temp_volatility
0    2025-12             9.95             4.0             4.62


/var/folders/xm/83xgwcz97ms1ltm3k7xl4dq80000gn/T/ipykernel_44282/1878629051.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  temp_features = df.groupby('year_month').apply(lambda x: pd.Series({


## 5. Precipitation Features

Engineer rain-related features that may impact flight delays.
* Number of rainy days
* Number of days with heavy rain (> 10 mm according to BOM and ICAO)
* Average amount of rain on rainy days

In [22]:
# Precipitation feature engineering
precip_features = df.groupby('year_month').apply(lambda x: pd.Series({
    # Rainy days
    'rainy_days': (x['rain'] > 0).sum(),
    'heavy_rain_days': (x['rain'] > 10).sum(),  # >10mm considered heavy
    
    # Rain intensity on rainy days
    'avg_rainfall_on_rainy_days': x[x['rain'] > 0]['rain'].mean() if (x['rain'] > 0).any() else 0
})).reset_index()

# Left-join merge with monthly stats
monthly_stats = monthly_stats.merge(precip_features, on='year_month', how='left')

print("Precipitation features added:")
print(precip_features)

Precipitation features added:
  year_month  rainy_days  heavy_rain_days  avg_rainfall_on_rainy_days
0    2025-12         9.0              0.0                        2.38


/var/folders/xm/83xgwcz97ms1ltm3k7xl4dq80000gn/T/ipykernel_44282/1439826448.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  precip_features = df.groupby('year_month').apply(lambda x: pd.Series({


## 6. Wind features

In [23]:
# Wind feature engineering

wind_features = df.groupby('year_month').apply(lambda x: pd.Series({
    # High wind days (thresholds based on aviation standards)
    'days_high_wind': (x['wind_speed'] > 8).sum(),  # >8 m/s (~15 knots)
    
    # Wind variability
    'wind_speed_std': x['wind_speed'].std()
})).reset_index()

# Left-join merge with monthly stats
monthly_stats = monthly_stats.merge(wind_features, on='year_month', how='left')

print("Wind features added:")
print(wind_features)

Wind features added:
  year_month  days_high_wind  wind_speed_std
0    2025-12             1.0            1.26


/var/folders/xm/83xgwcz97ms1ltm3k7xl4dq80000gn/T/ipykernel_44282/1084947139.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wind_features = df.groupby('year_month').apply(lambda x: pd.Series({


## 7. Humidity features

In [24]:
# Humidity feature engineering
humidity_features = df.groupby('year_month').apply(lambda x: pd.Series({
    # High humidity days (potential for fog/low visibility)
    'days_high_humidity': (x['hum_max'] > 90).sum()
})).reset_index()

# Merge with main stats
monthly_stats = monthly_stats.merge(humidity_features, on='year_month', how='left')

print("Humidity features added:")
print(humidity_features)

Humidity features added:
  year_month  days_high_humidity
0    2025-12                   6


/var/folders/xm/83xgwcz97ms1ltm3k7xl4dq80000gn/T/ipykernel_44282/1556837375.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  humidity_features = df.groupby('year_month').apply(lambda x: pd.Series({


## 8. Extreme Weather Events

Composite features capturing "extreme" weather days that likely cause delays. Extreme weather days are defined to be the days where either max_temp > 35 C OR rain > 10 mm OR wind_speed > 8 m/s OR hum_max > 95% occurs.

In [25]:
# Extreme weather event features
extreme_features = df.groupby('year_month').apply(lambda x: pd.Series({
    # Extreme weather days
    'extreme_weather_days': ((x['temp_max'] > 35) | (x['rain'] > 10) | (x['wind_speed'] > 8) | (x['hum_max'] > 95)).sum()
})).reset_index()

# Merge with main stats
monthly_stats = monthly_stats.merge(extreme_features, on='year_month', how='left')

print("Extreme weather features added:")
print(extreme_features)

Extreme weather features added:
  year_month  extreme_weather_days
0    2025-12                     5


/var/folders/xm/83xgwcz97ms1ltm3k7xl4dq80000gn/T/ipykernel_44282/814623203.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  extreme_features = df.groupby('year_month').apply(lambda x: pd.Series({


## 10. Summary Statistics

In [ ]:
# Display complete monthly statistics
print("="*80)
print("MONTHLY WEATHER STATISTICS SUMMARY")
print("="*80)
print(f"\nTotal months analyzed: {len(monthly_stats)}")
print(f"Total features created: {len(monthly_stats.columns) - 2}")
print(f"\nFeature breakdown:")
print(f"  - Basic statistics: 10")
print("  - Temperature features: 3")
print("  - Precipitation features: 3")
print("  - Wind features: 2")
print("  - Humidity features: 1")
print("  - Extreme weather features: 1")
print(f"\nTotal: {len(monthly_stats.columns) - 2} features")

print("\n" + "="*80)
print("Complete monthly statistics:")
print(monthly_stats)

MONTHLY WEATHER STATISTICS SUMMARY

Total months analyzed: 1
Total features created: 20

Feature breakdown:
  - Basic statistics: 7
  - Temperature features: 3
  - Precipitation features: 3
  - Wind features: 2
  - Humidity features: 1
  - Extreme weather features: 1

Total: 20 features

Complete monthly statistics:
  year_month  date_count  max_temperature  avg_max_temp  min_temperature  \
0    2025-12          31             42.6         28.28             13.4   

   avg_min_temp  max_daily_rainfall  avg_rainfall_per_day  avg_wind_speed  \
0         18.33                 5.8                  0.69            6.02   

   max_wind_speed  avg_max_humidity  avg_min_humidity  temp_range_mean  \
0             8.1             80.87             38.26             9.95   

   days_above_35C  temp_volatility  rainy_days  heavy_rain_days  \
0             4.0             4.62         9.0              0.0   

   avg_rainfall_on_rainy_days  days_high_wind  wind_speed_std  \
0                        

## 11. Export Results

Save the computed monthly statistics for use in modeling.

In [31]:
# Export to CSV
output_path = '../data/processed/monthly_weather_stats_single.csv'
monthly_stats.to_csv(output_path, index=False)
print(f"Monthly statistics exported to: {output_path}")

# Display column list for reference
print("\nAll features available for modeling:")
print("="*80)
for i, col in enumerate(monthly_stats.columns[2:], 1):
    print(f"{i:2d}. {col}")

Monthly statistics exported to: ../data/processed/monthly_weather_stats_single.csv

All features available for modeling:
 1. max_temperature
 2. avg_max_temp
 3. min_temperature
 4. avg_min_temp
 5. max_daily_rainfall
 6. avg_rainfall_per_day
 7. avg_wind_speed
 8. max_wind_speed
 9. avg_max_humidity
10. avg_min_humidity
11. temp_range_mean
12. days_above_35C
13. temp_volatility
14. rainy_days
15. heavy_rain_days
16. avg_rainfall_on_rainy_days
17. days_high_wind
18. wind_speed_std
19. days_high_humidity
20. extreme_weather_days
